# 02 - Data Cleaning (all fields + IT deep dive)

## Objective
Clean `jobs_raw.csv`, give every job a **market field** (IT, Marketing, Sales, Design...), label the IT jobs with their **IT field** (Backend, DevOps, Data...), and detect the IT skills.

**Rules used in this notebook**
- The raw file is never edited. All cleaning happens on `df`, and the result is saved as a new file.
- Every cell can be re-run safely (no cell breaks if you run it twice).
- `description` stays a **string**, never a list.

In [17]:
import ast
import html
import re

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

raw_path = "../data/raw/jobs_raw.csv"
clean_path = "../data/clean/jobs_clean.csv"

df = pd.read_csv(raw_path, encoding="utf-8")   # always start from the raw file
print(df.shape)
df.dtypes

(117, 14)


id               object
date             object
company          object
position         object
tags             object
description      object
location         object
salary_min      float64
salary_max      float64
apply_url        object
url              object
source           object
category         object
collected_at     object
dtype: object

## 1. Duplicates

In [18]:
before = len(df)
df = df.drop_duplicates(subset="id")
df = df.drop_duplicates(subset="url")
# the same job can be posted twice with different ids
df = df.drop_duplicates(subset=["company", "position", "location"], keep="first")
print("removed:", before - len(df), "| left:", len(df))

removed: 0 | left: 117


## 2. Fix broken characters (encoding)
Text such as `MecÃ¡nico` or `Â\xa0` is UTF-8 text that was decoded as latin-1.
The fix is applied to **all** text columns, not only `location`. If a text cannot be repaired it is left unchanged.

In [19]:
def fix_mojibake(text):
    if not isinstance(text, str):
        return text
    for enc in ("latin-1", "cp1252"):
        try:
            return text.encode(enc).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return text            # already fine, or cannot be repaired

for col in ["company", "position", "location", "description"]:
    df[col] = df[col].apply(fix_mojibake)

# check what is still broken (should be empty or very small)
still_broken = df[df["position"].str.contains("[ÃÂ]", regex=True, na=False)]
still_broken[["position", "company"]]

,position,company


## 3. Text columns: company, position

In [20]:
for col in ["company", "position"]:
    df[col] = (df[col].astype("string")
               .str.replace(r"\s+", " ", regex=True)
               .str.strip())
df[["company", "position"]].isna().sum()

company     0
position    0
dtype: int64

## 4. Location
Steps: fill missing, remove trailing commas, merge `Remoto` -> `Remote`, and remove repeated parts
(`Austin, Austin, Texas, United States` -> `Austin, Texas, United States`).

In [21]:
def clean_location(text):
    if pd.isna(text):
        return "Unknown"
    text = re.sub(r"\s+", " ", str(text)).strip().strip(",").strip()
    if not text:
        return "Unknown"
    parts, seen = [], set()
    for part in [p.strip() for p in text.split(",")]:
        if part and part.lower() not in seen:     # drop repeated parts
            seen.add(part.lower())
            parts.append(part)
    text = ", ".join(parts)
    return re.sub(r"(?i)\bremoto\b", "Remote", text)

df["location"] = df["location"].apply(clean_location)
df["location"].value_counts().head(10)

location
Unknown                                  37
Remote                                    6
Worldwide                                 5
Northern America, LATAM, Europe, APAC     3
Austin, Texas, United States              2
Remote - US                               2
United States                             2
Boston                                    2
Americas, Europe, Israel                  2
Redwood City                              2
Name: count, dtype: int64

## 5. Date

In [22]:
df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True).dt.tz_localize(None)
df = df.dropna(subset=["date"])
df["posted_date"] = df["date"].dt.normalize()          # date without time
df["year_month"] = df["date"].dt.to_period("M").astype(str)
print(df["posted_date"].min(), "->", df["posted_date"].max())

2026-07-30 00:00:00 -> 2026-10-05 00:00:00


## 6. Salary
`0` means "not provided", not a real salary. RemoteOK salaries are yearly (USD).

In [23]:
df[["salary_min", "salary_max"]] = df[["salary_min", "salary_max"]].replace(0, np.nan)

swap = df["salary_min"] > df["salary_max"]                # min and max reversed
df.loc[swap, ["salary_min", "salary_max"]] = df.loc[swap, ["salary_max", "salary_min"]].values

df["salary_mid"] = df[["salary_min", "salary_max"]].mean(axis=1)
unrealistic = (df["salary_mid"] < 10_000) | (df["salary_mid"] > 500_000)
df.loc[unrealistic, ["salary_min", "salary_max", "salary_mid"]] = np.nan

df["salary_available"] = df["salary_mid"].notna()
print("jobs with salary:", int(df["salary_available"].sum()), "of", len(df))

jobs with salary: 22 of 117


## 7. Tags
In the CSV, tags are saved as text (`"['a', 'b']"`). This function works for text **and** for real lists,
so running the cell twice does not cause `malformed node or string`.

In [ ]:
def parse_tags(value):
    if isinstance(value, list):
        items = value
    elif isinstance(value, str):
        try:
            items = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            items = []
    else:
        items = []
    return [str(t).strip().lower() for t in items]

df["tags"] = df["tags"].apply(parse_tags)
df["tags_text"] = df["tags"].apply(" ".join)
df["tags"].explode().value_counts().head(10)


tags_text
                                                                                                                                                                                                                        2
golang dev engineer digital nomad                                                                                                                                                                                       2
front end design react graphql react.js technical web dev javascript css chatbot html typescript node node.js exec sys admin engineer backend digital nomad                                                             2
golang infosec engineer part time                                                                                                                                                                                       1
exec design architecture recruiter customer support marketing content writing ops golang full time digital nomad dev s

## 8. Description
Two bugs fixed here:
1. The old code wrapped the text in `[ ... ]`, so the column became a list.
2. `getText("")` glued words together (`cycledecision`), which breaks skill search. The separator must be a space.

In [25]:
def html_to_text(raw):
    if not isinstance(raw, str):
        return ""
    text = BeautifulSoup(raw, "html.parser").get_text(" ", strip=True)
    text = html.unescape(text)
    return re.sub(r"\s+", " ", text).strip()

df["description_clean"] = df["description"].apply(html_to_text)
df = df[df["description_clean"] != ""]

assert df["description_clean"].apply(type).eq(str).all()
print("html tags left:", df["description_clean"].str.contains("<[^>]+>", regex=True).sum())
df["description_clean"].iloc[0][:300]

html tags left: 3


'Remote - Location: Mostly remote (within Germany), with monthly in-person collaboration at our Berlin officeHours: Part-time, 32 hours per weekContract: Fixed-termSalary: EUR 53,000 – 59,000 (at 32 hours per week)We are building a new capability that turns fragmented,... Please mention the word **TH'

## 9. Search text
One lowercase text per job (position + tags + description) used to look for skills.
Notice the spaces between the three parts.

In [26]:
df["search_text"] = (df["position"] + " "
                     + df["tags_text"] + " "
                     + df["description_clean"]).str.lower()
df["search_text"].str.len().describe()

count          117.0
mean     4151.547009
std      3126.368529
min            596.0
25%           2080.0
50%           3753.0
75%           5126.0
max          25292.0
Name: search_text, dtype: Float64

## 10. IT field
The raw data contains all job types (sales, marketing, design...). Each job gets a **field** from its title.
- Titles that are clearly not IT are excluded first (`Sales Engineer`, `Marketing Manager`...).
- Fields are checked **in order**; the first match wins (so `Data Engineer` is Data, not Backend).
- If the title says nothing, the job tags are used as a second chance.
- Everything else stays `Other`. You can edit the patterns below and re-run.

In [27]:
NOT_IT = (r"sales|account executive|account manager|customer success|customer service|recruit|talent|marketing"
          r"|copywrit|content writer|accountant|bookkeep|virtual assistant|designer|business development"
          r"|mechanical|electrical|civil|chemical|automotive|manufacturing|nurse|teacher|driver|paralegal")

FIELD_PATTERNS = {                       # checked in this order
    "AI / Machine Learning": r"machine learning|\bml\b|\bai\b|artificial intelligence|deep learning|\bllm\b|\bnlp\b|computer vision",
    "Data": r"data (?:scien|analy|engineer|architect|platform)|analytics|business intelligence|\bbi\b|\betl\b|database|\bdba\b|big data",
    "Security": r"security|cyber|infosec|penetration|pentest|\bsoc\b|appsec|devsecops",
    "DevOps / Cloud": r"devops|\bsre\b|site reliability|platform engineer|cloud|infrastructure|kubernetes|release engineer",
    "QA / Testing": r"\bqa\b|quality assurance|\btesters?\b|\btesting\b|\bsdet\b|test automation",
    "IT Support / SysAdmin": (r"it support|help ?desk|service desk|technical support|support engineer|desktop support|sysadmin"
                              r"|systems? administrator|network (?:engineer|administrator)"
                              r"|it (?:administrator|manager|specialist|technician|operations)"),
    "Mobile": r"\bios\b|android|mobile|flutter|react native|swift|kotlin",
    "Full Stack": r"full[- ]?stack",
    "Frontend": r"front[- ]?end|\breact\b|\bvue\b|angular|ui engineer|web developer",
    "Backend": r"back[- ]?end|\bapi\b|python|\bjava\b|golang|\bgo (?:developer|engineer)|ruby|rails|\bphp\b|node\.?js|\bnode\b|\.net|c#|rust|scala|elixir|laravel|django",
    "Software Engineering (General)": r"software|developer|programmer|engineer|architect|\bswe\b|tech lead|technical lead|coder",
}

TAG_RESCUE = {"dev": "Software Engineering (General)", "devops": "DevOps / Cloud", "sys admin": "IT Support / SysAdmin",
              "security": "Security", "front end": "Frontend", "backend": "Backend",
              "full stack": "Full Stack", "machine learning": "AI / Machine Learning"}

def classify(title, tags):
    t = title.lower()
    if re.search(NOT_IT, t):
        return "Other"
    for name, pattern in FIELD_PATTERNS.items():
        if re.search(pattern, t):
            return name
    for tag in tags:
        if tag in TAG_RESCUE:
            return TAG_RESCUE[tag]
    return "Other"

df["field"] = [classify(title, tags) for title, tags in zip(df["position"], df["tags"])]

# Remotive tells us the category of its jobs: use it to rescue IT jobs with unclear titles
if "category" not in df.columns:
    df["category"] = np.nan
IT_CATEGORY = r"software|devops|sysadmin|\bqa\b|quality assurance|\bdata\b|artificial|information.technology|cyber|security"
in_it_category = df["category"].fillna("").str.lower().str.contains(IT_CATEGORY, regex=True)
not_it_title = df["position"].str.lower().str.contains(NOT_IT, regex=True)
rescue = (df["field"] == "Other") & in_it_category & ~not_it_title
df.loc[rescue, "field"] = "Software Engineering (General)"
print("jobs rescued by Remotive category:", int(rescue.sum()))

df["is_it_job"] = df["field"] != "Other"
print(df["field"].value_counts())

# review by eye: titles that were NOT classified as IT (look for IT jobs the rules missed)
df.loc[~df["is_it_job"], "position"].drop_duplicates().head(30).to_frame()

jobs rescued by Remotive category: 0
field
Other                             49
Software Engineering (General)    28
AI / Machine Learning             11
IT Support / SysAdmin              7
Frontend                           4
Data                               4
Backend                            4
Full Stack                         4
Security                           2
DevOps / Cloud                     2
QA / Testing                       1
Mobile                             1
Name: count, dtype: int64


,position
1,Federal Business Development Director
2,Head of Operations
3,Junior Crypto Analyst & Trader
4,Enterprise Sales Development Representative
5,Client Advisor
7,Danish Speaking Solutions Consultant Work Sofi...
8,Mecánico Automotriz Diagnóstico y Presupuestos
9,Video Data Annotator
13,Junior Digital Assets Operations Analyst
15,Freelance grabación de tareas cotidianas para ...


## 10b. Market field (all jobs)
Every job gets a general field so we can compare IT with the rest of the market.
1. IT jobs (found above) -> `IT & Software`.
2. Otherwise the **Remotive category** is used when it exists (it is more reliable than guessing).
3. Otherwise (RemoteOK jobs) the title is used.
4. Anything left is `Other`.

In [28]:
FIELD_RULES = [        # (market field, pattern on the Remotive category, pattern on the job title)
    ("Customer Support",            r"customer",                r"customer (?:support|success|service|experience)|support (?:specialist|agent|representative)|chat support"),
    ("Sales",                       r"sales",                   r"sales|account executive|business development|\bsdr\b|\bbdr\b|account manager"),
    ("Marketing",                   r"marketing",               r"marketing|\bseo\b|growth|social media|community manager|brand|ppc"),
    ("Design",                      r"design",                  r"designer|\bux\b|\bui\b|graphic|illustrat|animator|video editor"),
    ("Product & Project Management", r"product|project",        r"product manager|project manager|program manager|scrum master|product owner"),
    ("Finance & Legal",             r"finance|legal|accounting", r"account(?:ant|ing)|bookkeep|financ|\bcpa\b|payroll|legal|attorney|paralegal|compliance|auditor|\btax\b"),
    ("HR & Recruiting",             r"human|\bhr\b|recruit",   r"recruit|talent|\bhr\b|human resources|people ops"),
    ("Writing & Content",           r"writing|content",         r"writer|editor|content|translator|journalist|copywrit"),
    ("Education",                   r"education|teach",         r"teacher|tutor|instructor|lecturer|curriculum"),
    ("Healthcare",                  r"medical|health",          r"nurse|medical|clinical|physician|therapist|health"),
    ("Operations & Business",       r"business|operations|management", r"operations|administrat|assistant|coordinator|office|executive"),
]

def market_field(title, category, is_it):
    if is_it:
        return "IT & Software"
    category = category.lower() if isinstance(category, str) else ""
    if category:
        for name, cat_pattern, _ in FIELD_RULES:
            if re.search(cat_pattern, category):
                return name
    t = title.lower()
    for name, _, title_pattern in FIELD_RULES:
        if re.search(title_pattern, t):
            return name
    return "Other"

df["market_field"] = [market_field(t, c, i) for t, c, i in zip(df["position"], df["category"], df["is_it_job"])]
print(df["market_field"].value_counts())

# review by eye: titles left as "Other"
df.loc[df["market_field"] == "Other", "position"].drop_duplicates().head(30).to_frame()

market_field
IT & Software                   68
Other                           18
Sales                            7
Operations & Business            5
Marketing                        5
Customer Support                 4
Design                           4
HR & Recruiting                  2
Writing & Content                2
Product & Project Management     1
Finance & Legal                  1
Name: count, dtype: int64


,position
3,Junior Crypto Analyst & Trader
5,Client Advisor
7,Danish Speaking Solutions Consultant Work Sofi...
8,Mecánico Automotriz Diagnóstico y Presupuestos
9,Video Data Annotator
15,Freelance grabación de tareas cotidianas para ...
36,External Data Specialist
38,Quality Dispense Technician North
50,Digital Workplace & Automation Specialist
51,Solutions Delivery Manager


## 11. Seniority

In [29]:
def seniority(title):
    t = title.lower()
    if re.search(r"\b(intern|internship|trainee)\b", t):                           return "Intern"
    if re.search(r"\b(junior|jr|entry|associate)\b", t):                           return "Junior"
    if re.search(r"\b(senior|sr|lead|principal|staff|head|director|manager)\b", t): return "Senior+"
    return "Mid"

df["seniority"] = df["position"].apply(seniority)
df["seniority"].value_counts()

seniority
Mid        70
Senior+    41
Junior      6
Name: count, dtype: int64

## 12. Skills
One True/False column per skill (`has_python`, `has_docker`...), searched in the title, tags and description.
Patterns also catch variants (`k8s`, `postgres`, `github actions`). Add or remove skills freely.

In [30]:
SKILL_PATTERNS = {
    # languages
    "python": r"\bpython\b",
    "javascript": r"javascript|\bjs\b|ecmascript",
    "typescript": r"typescript",
    "java": r"\bjava\b",
    "csharp_dotnet": r"c#|\.net\b|dotnet|asp\.net",
    "cpp": r"c\+\+",
    "go": r"\bgolang\b|\bgo (?:programming|language|developer|engineer)\b",
    "rust": r"\brust\b",
    "php": r"\bphp\b|laravel|symfony",
    "ruby": r"\bruby\b|\brails\b",
    "kotlin_swift": r"\bkotlin\b|\bswift(?:ui)?\b",
    "sql": r"\b(?:sql|mysql|postgres(?:ql)?|t-sql|sqlite)\b",
    # frameworks
    "react": r"\breact(?:\.?js)?\b(?!\s+(?:to|quickly|fast))",
    "angular": r"\bangular\b",
    "vue": r"\bvue(?:\.?js)?\b",
    "nodejs": r"node\.?js|\bexpress\.?js\b|\bnestjs\b",
    "django_flask": r"\bdjango\b|\bflask\b|\bfastapi\b",
    "spring": r"spring boot|spring framework",
    # cloud and devops
    "aws": r"\baws\b|amazon web services",
    "azure": r"\bazure\b",
    "gcp": r"\bgcp\b|google cloud",
    "docker": r"\bdocker\b",
    "kubernetes": r"kubernetes|\bk8s\b",
    "terraform_iac": r"terraform|infrastructure as code|\bansible\b|pulumi",
    "ci_cd": r"ci/cd|ci-cd|continuous integration|continuous delivery|github actions|jenkins|gitlab ci",
    "linux": r"\blinux\b|\bunix\b",
    "git": r"\bgit\b|github|gitlab|bitbucket",
    # data and databases
    "mongodb_nosql": r"mongo ?db|\bnosql\b",
    "redis": r"\bredis\b",
    "kafka_queues": r"\bkafka\b|rabbitmq",
    "spark_hadoop": r"\b(?:py)?spark\b|\bhadoop\b|databricks",
    "cloud_warehouse": r"snowflake|bigquery|redshift",
    "power_bi": r"power ?bi",
    "tableau": r"\btableau\b",
    "excel": r"\bexcel\b",
    # AI
    "machine_learning": r"machine learning|\bml\b|scikit|sklearn",
    "deep_learning": r"deep learning|pytorch|tensorflow|keras",
    "llm_genai": r"\bllms?\b|generative ai|\bgenai\b|large language|openai|chatgpt|langchain",
    # practices
    "rest_api": r"rest(?:ful)? ?api|\bapis?\b|graphql",
    "microservices": r"microservices?",
    "testing": r"unit test|integration test|\bjest\b|pytest|selenium|cypress|playwright|test automation",
    "agile": r"\bagile\b|\bscrum\b|\bkanban\b",
    "security_practices": r"owasp|penetration test|\bsiem\b|vulnerabilit|encryption|\biam\b",
}

for skill, pattern in SKILL_PATTERNS.items():
    df["has_" + skill] = df["search_text"].str.contains(pattern, regex=True)

skill_cols = ["has_" + s for s in SKILL_PATTERNS]

## 13. Quick look
IT skills are counted on **IT jobs only** (the skill list is technical, so it does not describe marketing or sales jobs).

In [31]:
it_jobs = df[df["is_it_job"]]
print("All jobs:", len(df), "| IT jobs:", len(it_jobs), f"({len(it_jobs) / len(df):.0%})")
if len(it_jobs) < 300:
    print("NOTE: fewer than 300 IT jobs. Small IT fields will have too few jobs for reliable percentages.")

print(df["market_field"].value_counts().to_string())
(it_jobs[skill_cols].mean() * 100).round(1).sort_values(ascending=False).head(15)

All jobs: 117 | IT jobs: 68 (58%)
NOTE: fewer than 300 IT jobs. Small IT fields will have too few jobs for reliable percentages.
market_field
IT & Software                   68
Other                           18
Sales                            7
Operations & Business            5
Marketing                        5
Customer Support                 4
Design                           4
HR & Recruiting                  2
Writing & Content                2
Product & Project Management     1
Finance & Legal                  1


has_go                  42.6
has_rest_api            26.5
has_typescript          20.6
has_javascript          20.6
has_machine_learning    20.6
has_python              16.2
has_llm_genai           16.2
has_react               16.2
has_ruby                14.7
has_sql                 14.7
has_nodejs              14.7
has_excel               14.7
has_java                11.8
has_angular             10.3
has_csharp_dotnet       10.3
dtype: Float64

## 14. Final checks and save

In [32]:
assert df["id"].is_unique
assert df["description_clean"].apply(type).eq(str).all()
assert df[["company", "position", "date"]].notna().all().all()
assert df["location"].notna().all()

final = df.drop(columns=["description", "apply_url"]).copy()
final["tags"] = final["tags"].apply("|".join)          # list -> "a|b|c" so the CSV stays clean
final = final.reset_index(drop=True)

final.to_csv(clean_path, index=False, encoding="utf-8")
print("saved:", clean_path, final.shape)

saved: ../data/clean/jobs_clean.csv (117, 66)
